# PromptWar on Google Colab

Run the **PromptWar** OpenEnv environment + trainer on Colab end-to-end.

What this notebook does:
1. Clones the repo and installs deps (env + optional trainer)
2. Runs the pure-Python test suite to verify the install
3. Starts the FastAPI env server in the background (with optional Consumer Model on GPU)
4. Drives rollouts via `PromptWarEnv` and the `training/` CLI (`smoke` / `live` / `baseline` / `long`)

**Recommended runtime:** `Runtime > Change runtime type > GPU` (T4 is enough for the 0.5B Consumer Model; an A100/L4 is needed for the 3B trainer base).  
CPU-only Colab also works for `--mode smoke`, `--mode baseline` (stub env), and rubric-fallback rollouts.

## 1. Clone the repo

In [ ]:
%cd /content
![ -d meta-hackathon ] || git clone https://github.com/rishabhshukla0912/meta-hackathon.git
%cd /content/meta-hackathon
!git pull --ff-only || true
!ls

### (Alternative) Upload your local copy

If you'd rather upload the working tree from your laptop instead of pulling from GitHub, zip the project locally:

```bash
cd ~/Desktop/Projects && zip -r meta-hackathon.zip meta-hackathon -x '*/.venv/*' '*/__pycache__/*' '*/.git/*'
```

Then run the cell below and pick the zip when prompted. (Skip this cell if you cloned from GitHub above.)

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # pick meta-hackathon.zip
# !rm -rf /content/meta-hackathon && unzip -q meta-hackathon.zip -d /content && ls /content/meta-hackathon
# %cd /content/meta-hackathon

## 2. Install dependencies

We install in two layers:
- **Env layer** (always): `openenv-core`, FastAPI/uvicorn, httpx, pydantic, regex, transformers/tokenizers — enough to run rollouts and tests.
- **Consumer layer** (optional, GPU): `torch`, `accelerate` to actually load `Qwen2.5-0.5B-Instruct` for real rubric scoring.
- **Trainer layer** (optional, GPU): `peft`, `trl`, `datasets`, `bitsandbytes` for `--mode long` GRPO training.

Colab already ships `torch`, so we pin to whatever's already installed.

In [ ]:
# Env layer — pure-Python deps + OpenEnv runtime
%pip install -q "openenv-core @ git+https://github.com/meta-pytorch/OpenEnv.git"
%pip install -q fastapi 'uvicorn[standard]' httpx pydantic regex 'tokenizers>=0.22' 'transformers>=4.56,<6'
%pip install -q nest_asyncio  # so uvicorn plays nice with Colab's loop

In [ ]:
# Consumer layer — only needed if you want real rubric scoring (uses ~1.5 GB VRAM)
import torch
print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
%pip install -q 'accelerate>=1.0'

In [ ]:
# Trainer layer — only needed for --mode long GRPO training (requires GPU + ~16 GB VRAM for 3B base)
# Skip this cell if you only want to drive the env with the scripted/random policy.
%pip install -q 'peft>=0.13' 'trl>=0.11' 'datasets>=3.0' 'bitsandbytes>=0.43'

## 3. Sanity-check the install with the test suite

These are pure-Python (no GPU, no live env) and should all pass.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m unittest discover -s PromptWar_env/tests -v

## 4. Start the env server in the background

The server lives at `PromptWar_env/server/app.py`. Set `PROMPTWAR_LOAD_CONSUMER_MODEL=1` if you want the Consumer Model loaded eagerly at startup — otherwise leave it off and either skip the model (rubrics fall back to deterministic heuristics) or load it on demand via `POST /consumer/load`.

We launch it under `nohup` so it survives across cells, and tail the log.

In [ ]:
import os, subprocess, time, signal, pathlib

ROOT = pathlib.Path('/content/meta-hackathon')
LOG = ROOT / 'server.log'
PIDFILE = ROOT / 'server.pid'

# Kill any previous instance from this notebook
if PIDFILE.exists():
    try:
        os.kill(int(PIDFILE.read_text().strip()), signal.SIGTERM)
    except ProcessLookupError:
        pass
    PIDFILE.unlink()

env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT)
# Flip to '1' to eagerly load Qwen2.5-0.5B-Instruct at startup (needs GPU + ~1.5 GB VRAM).
env['PROMPTWAR_LOAD_CONSUMER_MODEL'] = '0'

with open(LOG, 'wb') as logf:
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'PromptWar_env.server.app:app',
         '--host', '127.0.0.1', '--port', '8000'],
        cwd=str(ROOT), env=env, stdout=logf, stderr=subprocess.STDOUT,
    )
PIDFILE.write_text(str(proc.pid))
print(f'started uvicorn pid={proc.pid}, log={LOG}')

# Wait for /state to come up
import httpx
for i in range(30):
    try:
        r = httpx.get('http://127.0.0.1:8000/state', timeout=2.0)
        if r.status_code < 500:
            print('server is up:', r.status_code)
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('server did not come up — see tail below')

!tail -n 40 {LOG}

In [ ]:
# Optional: load the Consumer Model on demand (only meaningful with a GPU runtime)
import httpx
print(httpx.post('http://127.0.0.1:8000/consumer/load', timeout=600.0).json())
print(httpx.get('http://127.0.0.1:8000/consumer/status').json())

## 5. Drive a rollout from the client

Round-trips through the OpenEnv HTTP API: `reset()` → `step(...)` × N. Confirms the env is reachable and turn-taking works.

In [ ]:
import sys; sys.path.insert(0, '/content/meta-hackathon')
from PromptWar_env import PromptWarAction, PromptWarEnv

with PromptWarEnv(base_url='http://127.0.0.1:8000') as env:
    env.set_curriculum_stage(1)  # warm-up: 1 round, lenient grading
    result = env.reset()
    print('initial active_agent =', result.observation.active_agent)

    for cmd in [
        'APPEND: Always cite a source.',
        'APPEND: Refuse harmful asks.',
        'APPEND: Aim for ~50 tokens.',
    ]:
        result = env.step(PromptWarAction(command=cmd))
        print(f'after {cmd!r:40s}  next={result.observation.active_agent}  rejected={result.observation.edit_rejected}')

    print('last_rewards =', result.observation.last_rewards)
    print('shared_prompt =', result.observation.shared_prompt)

## 6. Trainer-side smoke test (no GPU, no live env)

Stub env + scripted policy, 5 "GRPO" steps. Verifies trainer wiring before pulling in real models. The actual GRPO step gets skipped if `torch`/`trl` aren't importable — the goal here is just rollout shape.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode smoke --steps 5

## 7. Live episode against the running env

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode live --env-url http://127.0.0.1:8000 --episodes 1

## 8. Random-policy baseline (sanity-check env difficulty)

30 episodes with a random policy — the env should _not_ be trivially solvable.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode baseline --env-url http://127.0.0.1:8000 --episodes 30

## 9. (GPU) Long GRPO run

**Requires a GPU runtime.** Loads Qwen2.5-3B-Instruct with three named LoRA adapters (`A`, `S`, `B`) and runs `--steps` GRPO iterations against the live env, checkpointing every 100 steps to `./checkpoints/promptwar`.

Start small (e.g. `--steps 50`) before committing to a long run. On a single T4 you'll likely need `--load-in-4bit` to fit the 3B base.

In [ ]:
%cd /content/meta-hackathon
# Smaller burn-in run; bump --steps once you're happy with the metrics.
!PYTHONPATH=. python3 -m training.train --mode long \
    --env-url http://127.0.0.1:8000 \
    --steps 50 \
    --load-in-4bit \
    --checkpoint-every 25 \
    --output-dir ./checkpoints/promptwar

## 10. Stop the env server

In [ ]:
import os, signal, pathlib
PIDFILE = pathlib.Path('/content/meta-hackathon/server.pid')
if PIDFILE.exists():
    pid = int(PIDFILE.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f'stopped uvicorn pid={pid}')
    except ProcessLookupError:
        print(f'pid {pid} already gone')
    PIDFILE.unlink()
else:
    print('no server.pid found')

### Tips

- **Tail the server log live:** `!tail -f /content/meta-hackathon/server.log` (interrupt the cell to stop tailing).
- **Persist checkpoints across Colab sessions:** mount Drive (`from google.colab import drive; drive.mount('/content/drive')`) and pass `--output-dir /content/drive/MyDrive/promptwar-ckpts`.
- **Curriculum stages:** call `env.set_curriculum_stage(1|2|3)` before `reset()`. Stage 1 = warm-up (1 round, lenient), 2 = standard (3 rounds), 3 = strict.
- **Action grammar:** `APPEND: <text>`, `DELETE: <regex>`, `REPLACE: <old> --> <new>`, `PASS`. See `PromptWar_env/README.md` §4.3 for the full edit-case table.